# Module 4: Database Integration & SQL Analysis
This notebook sets up the SQLite database schema (`aistockwave.db`), populates it with our wrangled datasets, and executes analytical SQL queries.

In [ ]:
import sqlite3
import pandas as pd

print("SQLite3 database engine initialized.")

### 1. Establishing Schema Tables
We construct tables: `users`, `stocks`, `portfolio`, `transactions`, `watchlist`, and `news`.

In [ ]:
conn = sqlite3.connect("aistockwave.db")
cursor = conn.cursor()

# 1. Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    balance REAL DEFAULT 100000.0,
    is_admin INTEGER DEFAULT 0
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS stocks (
    symbol TEXT PRIMARY KEY,
    company_name TEXT NOT NULL,
    current_price REAL NOT NULL,
    open_price REAL,
    close_price REAL,
    high_price REAL,
    low_price REAL,
    volume INTEGER,
    market_cap REAL,
    pe_ratio REAL,
    eps REAL,
    dividend_yield REAL,
    high_52week REAL,
    low_52week REAL
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS portfolio (
    user_id INTEGER,
    symbol TEXT,
    quantity INTEGER NOT NULL DEFAULT 0,
    avg_buy_price REAL NOT NULL DEFAULT 0.0,
    PRIMARY KEY (user_id, symbol)
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    type TEXT NOT NULL,
    symbol TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    price REAL NOT NULL,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS watchlist (
    user_id INTEGER,
    symbol TEXT,
    PRIMARY KEY (user_id, symbol)
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS news (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    description TEXT,
    source TEXT,
    date TEXT
)""")

conn.commit()
print("Database tables created successfully.")

### 2. Loading Wrangled Data into the Database Tables
We populate tables from the exported CSV sheets.

In [ ]:
# Load CSV files
df_stocks_clean = pd.read_csv("cleaned_stocks_data.csv")
df_news_clean = pd.read_csv("cleaned_news_data.csv")

# Insert Stocks
df_stocks_clean.columns = [
    'symbol', 'company_name', 'current_price', 'open_price', 'close_price',
    'high_price', 'low_price', 'volume', 'market_cap', 'pe_ratio', 'eps',
    'dividend_yield', 'high_52week', 'low_52week'
]
df_stocks_clean.to_sql("stocks", conn, if_exists="replace", index=False)

# Insert News
df_news_clean.columns = ['title', 'description', 'source', 'date']
df_news_clean.to_sql("news", conn, if_exists="replace", index=False)

# Seed standard user accounts if not present
cursor.execute("SELECT COUNT(*) FROM users")
if cursor.fetchone()[0] == 0:
    cursor.execute("INSERT INTO users (full_name, email, password, balance, is_admin) VALUES ('Paper Trader', 'user@aistockwave.com', 'user123', 100000.0, 0)")
    cursor.execute("INSERT INTO users (full_name, email, password, balance, is_admin) VALUES ('System Administrator', 'admin@aistockwave.com', 'admin123', 100000.0, 1)")
    conn.commit()
    print("Demo users seeded.")

print("Tables seeded successfully.")

### 3. Running SQL Queries for Stock Analytics
Let's execute query checks on our SQLite database.

In [ ]:
# Query 1: Top 5 stocks by Market Cap
query_mc = """
SELECT symbol, company_name, market_cap, current_price 
FROM stocks 
ORDER BY market_cap DESC 
LIMIT 5
"""
print(pd.read_sql_query(query_mc, conn))

print("\n")

# Query 2: Stocks with high P/E multiples (valued highly)
query_pe = """
SELECT symbol, company_name, pe_ratio 
FROM stocks 
WHERE pe_ratio > 30.0
"""
print(pd.read_sql_query(query_pe, conn))

conn.close()